In [0]:
# =============================================================================
# CREATE SAMPLE DATAFRAMES FOR JOIN DEMONSTRATIONS
# =============================================================================
# We'll create two DataFrames:
#   1. employees_df: Contains employee information with dept_id as the join key
#   2. departments_df: Contains department information with dept_id as the join key
#
# Key observations for join demonstrations:
#   - Employee Alice (dept_id=10) and Bob (dept_id=20) have matching departments
#   - Employee Charlie (dept_id=30) has NO matching department (orphan record)
#   - Department 40 (Operations) has NO employees (orphan record)
#   - This setup allows us to see how different join types handle matching and non-matching records
# =============================================================================

# Employee data: (emp_id, emp_name, dept_id)
employees_data = [
    (1, "Alice", 10),
    (2, "Bob", 20),
    (3, "Charlie", 30)  # No matching department
]

# Department data: (dept_id, dept_name)
departments_data = [
    (10, "HR"),
    (20, "IT"),
    (40, "Operations")  # No matching employees
]

# Create DataFrames with explicit schemas
employees_df = spark.createDataFrame(employees_data, ["emp_id", "emp_name", "dept_id"])
departments_df = spark.createDataFrame(departments_data, ["dept_id", "dept_name"])

# Display both DataFrames to understand the source data
print("EMPLOYEES:")
display(employees_df)

print("\nDEPARTMENTS:")
display(departments_df)

In [0]:
# =============================================================================
# JOIN TYPE 1: INNER JOIN (Default Join)
# =============================================================================
# DEFINITION:
#   Inner join returns only the rows that have matching values in BOTH DataFrames.
#   If a row in one DataFrame doesn't have a match in the other, it's excluded.
#
# SYNTAX:
#   df1.join(df2, on="column_name", how="inner")
#   OR simply: df1.join(df2, on="column_name")  # "inner" is default
#
# USE CASE:
#   Use when you only want records that exist in both tables.
#   Example: Get employees who have a valid department assigned.
#
# EXPECTED RESULT:
#   - Alice (dept_id=10) + HR department ✓
#   - Bob (dept_id=20) + IT department ✓
#   - Charlie (dept_id=30) is EXCLUDED (no matching department)
#   - Operations department (dept_id=40) is EXCLUDED (no matching employees)
#   Result: 2 rows only
# =============================================================================

# Perform inner join on dept_id column
inner_join_result = employees_df.join(
    departments_df,
    on="dept_id",
    how="inner"  # Can also omit this parameter as "inner" is the default
)

print("INNER JOIN RESULT:")
print("Only rows with matching dept_id in both DataFrames")
display(inner_join_result)

In [0]:
# =============================================================================
# JOIN TYPE 2: LEFT JOIN (LEFT OUTER JOIN)
# =============================================================================
# DEFINITION:
#   Left join returns ALL rows from the LEFT DataFrame (employees_df),
#   and matching rows from the RIGHT DataFrame (departments_df).
#   If no match is found, the result contains NULL for right DataFrame columns.
#
# SYNTAX:
#   df1.join(df2, on="column_name", how="left")
#   OR: df1.join(df2, on="column_name", how="left_outer")
#
# USE CASE:
#   Use when you want all records from the left table, regardless of matches.
#   Example: Get all employees with their department info (if available).
#   Useful for finding orphan records (employees without departments).
#
# EXPECTED RESULT:
#   - Alice (dept_id=10) + HR department ✓
#   - Bob (dept_id=20) + IT department ✓
#   - Charlie (dept_id=30) + NULL for dept_name ✓ (employee kept, no match)
#   - Operations department (dept_id=40) is EXCLUDED (not in left DataFrame)
#   Result: 3 rows (all employees)
# =============================================================================

# Perform left join on dept_id column
left_join_result = employees_df.join(
    departments_df,
    on="dept_id",
    how="left"  # Can also use "left_outer"
)

print("LEFT JOIN RESULT:")
print("All employees, with department info where available")
print("Notice Charlie has NULL for dept_name (no matching department)")
display(left_join_result)

In [0]:
# =============================================================================
# JOIN TYPE 3: RIGHT JOIN (RIGHT OUTER JOIN)
# =============================================================================
# DEFINITION:
#   Right join returns ALL rows from the RIGHT DataFrame (departments_df),
#   and matching rows from the LEFT DataFrame (employees_df).
#   If no match is found, the result contains NULL for left DataFrame columns.
#
# SYNTAX:
#   df1.join(df2, on="column_name", how="right")
#   OR: df1.join(df2, on="column_name", how="right_outer")
#
# USE CASE:
#   Use when you want all records from the right table, regardless of matches.
#   Example: Get all departments with their employees (if any).
#   Useful for finding orphan records (departments without employees).
#
# EXPECTED RESULT:
#   - Alice (dept_id=10) + HR department ✓
#   - Bob (dept_id=20) + IT department ✓
#   - Operations (dept_id=40) + NULL for emp_id, emp_name ✓ (dept kept, no match)
#   - Charlie (dept_id=30) is EXCLUDED (not in right DataFrame)
#   Result: 3 rows (all departments)
# =============================================================================

# Perform right join on dept_id column
right_join_result = employees_df.join(
    departments_df,
    on="dept_id",
    how="right"  # Can also use "right_outer"
)

print("RIGHT JOIN RESULT:")
print("All departments, with employee info where available")
print("Notice Operations dept has NULL for emp_id and emp_name (no employees)")
display(right_join_result)

In [0]:
# =============================================================================
# JOIN TYPE 4: FULL OUTER JOIN
# =============================================================================
# DEFINITION:
#   Full outer join returns ALL rows from BOTH DataFrames.
#   When there's a match, it combines the rows.
#   When there's no match, it includes the row with NULL for missing columns.
#
# SYNTAX:
#   df1.join(df2, on="column_name", how="full")
#   OR: df1.join(df2, on="column_name", how="outer")
#   OR: df1.join(df2, on="column_name", how="full_outer")
#
# USE CASE:
#   Use when you want all records from both tables, showing all matches and mismatches.
#   Example: Get complete view of employees and departments, including orphans.
#   Useful for data quality checks and finding all unmatched records.
#
# EXPECTED RESULT:
#   - Alice (dept_id=10) + HR department ✓ (match)
#   - Bob (dept_id=20) + IT department ✓ (match)
#   - Charlie (dept_id=30) + NULL for dept_name ✓ (employee without dept)
#   - Operations (dept_id=40) + NULL for emp_id, emp_name ✓ (dept without employees)
#   Result: 4 rows (all records from both DataFrames)
# =============================================================================

# Perform full outer join on dept_id column
full_join_result = employees_df.join(
    departments_df,
    on="dept_id",
    how="full"  # Can also use "outer" or "full_outer"
)

print("FULL OUTER JOIN RESULT:")
print("All employees AND all departments, showing all matches and mismatches")
print("Notice both Charlie and Operations have NULLs where no match exists")
display(full_join_result)

In [0]:
# =============================================================================
# JOIN TYPE 5: LEFT SEMI JOIN
# =============================================================================
# DEFINITION:
#   Left semi join returns rows from the LEFT DataFrame that have a match in the
#   RIGHT DataFrame, but ONLY includes columns from the LEFT DataFrame.
#   It's like an INNER JOIN but only shows left table columns.
#
# SYNTAX:
#   df1.join(df2, on="column_name", how="left_semi")
#
# KEY DIFFERENCE FROM INNER JOIN:
#   - INNER JOIN: Returns matching rows with columns from BOTH DataFrames
#   - LEFT SEMI JOIN: Returns matching rows with columns from LEFT DataFrame ONLY
#
# USE CASE:
#   Use when you want to filter the left table based on existence in right table,
#   but don't need any columns from the right table.
#   Example: Get employees who have a department (filter), without showing dept info.
#   Similar to SQL's "WHERE EXISTS" clause.
#
# EXPECTED RESULT:
#   - Alice (dept_id=10) ✓ (has matching dept, shows emp_id, emp_name, dept_id only)
#   - Bob (dept_id=20) ✓ (has matching dept, shows emp_id, emp_name, dept_id only)
#   - Charlie (dept_id=30) is EXCLUDED (no matching department)
#   - NO dept_name column (only left DataFrame columns are included)
#   Result: 2 rows with 3 columns (emp_id, emp_name, dept_id)
# =============================================================================

# Perform left semi join on dept_id column
left_semi_join_result = employees_df.join(
    departments_df,
    on="dept_id",
    how="left_semi"
)

print("LEFT SEMI JOIN RESULT:")
print("Employees who have a department (filtered), showing only employee columns")
print("Notice: No dept_name column - only columns from employees_df")
display(left_semi_join_result)

In [0]:
# =============================================================================
# JOIN TYPE 6: LEFT ANTI JOIN
# =============================================================================
# DEFINITION:
#   Left anti join returns rows from the LEFT DataFrame that DO NOT have a match
#   in the RIGHT DataFrame. It's the opposite of LEFT SEMI JOIN.
#   Only includes columns from the LEFT DataFrame.
#
# SYNTAX:
#   df1.join(df2, on="column_name", how="left_anti")
#
# KEY DIFFERENCE FROM LEFT JOIN:
#   - LEFT JOIN: Returns all left rows, with NULL for non-matches
#   - LEFT ANTI JOIN: Returns ONLY non-matching left rows (orphans)
#
# USE CASE:
#   Use when you want to find orphan records (rows that don't have a match).
#   Example: Find employees without a valid department assignment.
#   Similar to SQL's "WHERE NOT EXISTS" clause.
#   Great for data quality checks and finding incomplete records.
#
# EXPECTED RESULT:
#   - Alice (dept_id=10) is EXCLUDED (has matching dept)
#   - Bob (dept_id=20) is EXCLUDED (has matching dept)
#   - Charlie (dept_id=30) ✓ (NO matching department - this is what we want!)
#   - NO dept_name column (only left DataFrame columns are included)
#   Result: 1 row with 3 columns (emp_id, emp_name, dept_id)
# =============================================================================

# Perform left anti join on dept_id column
left_anti_join_result = employees_df.join(
    departments_df,
    on="dept_id",
    how="left_anti"
)

print("LEFT ANTI JOIN RESULT:")
print("Employees WITHOUT a matching department (orphans)")
print("Notice: Only Charlie appears - he has no department in departments_df")
display(left_anti_join_result)

In [0]:
# =============================================================================
# JOIN TYPE 7: CROSS JOIN (CARTESIAN PRODUCT)
# =============================================================================
# DEFINITION:
#   Cross join returns the Cartesian product of two DataFrames.
#   Every row from the LEFT DataFrame is combined with EVERY row from the RIGHT DataFrame.
#   No join condition is needed or used.
#
# SYNTAX:
#   df1.crossJoin(df2)  # Special method, not using .join()
#   OR: df1.join(df2, how="cross")  # Alternative syntax
#
# MATHEMATICAL RESULT:
#   If left DataFrame has M rows and right DataFrame has N rows,
#   the result will have M × N rows.
#   In our case: 3 employees × 3 departments = 9 rows
#
# USE CASE:
#   Use with EXTREME CAUTION! Can produce huge result sets.
#   Common use cases:
#   - Generate all possible combinations (e.g., all employee-department pairs)
#   - Create a matrix or grid of values
#   - Fill missing data with all possible values
#   Example: Generate all possible employee-department assignment combinations.
#
# EXPECTED RESULT:
#   Every employee paired with every department:
#   - Alice + HR, Alice + IT, Alice + Operations
#   - Bob + HR, Bob + IT, Bob + Operations
#   - Charlie + HR, Charlie + IT, Charlie + Operations
#   Result: 9 rows (3 × 3 combinations)
#
# WARNING:
#   Cross join can be very expensive on large DataFrames!
#   1000 rows × 1000 rows = 1,000,000 rows!
# =============================================================================

# Perform cross join (no join condition needed)
cross_join_result = employees_df.crossJoin(departments_df)

print("CROSS JOIN RESULT:")
print("Every employee combined with every department (Cartesian product)")
print(f"3 employees × 3 departments = 9 total combinations")
display(cross_join_result)

# PySpark Join Types - Quick Reference Summary

## Join Types Comparison Table

| Join Type | Syntax | Returns | Columns Included | Use Case | Result Count (Our Example) |
|-----------|--------|---------|------------------|----------|---------------------------|
| **Inner Join** | `how="inner"` | Only matching rows from both DataFrames | Both left & right | Find records that exist in both tables | 2 rows (Alice, Bob) |
| **Left Join** | `how="left"` or `how="left_outer"` | All rows from left + matching from right | Both left & right (NULL for non-matches) | Keep all left records, add right info where available | 3 rows (Alice, Bob, Charlie) |
| **Right Join** | `how="right"` or `how="right_outer"` | All rows from right + matching from left | Both left & right (NULL for non-matches) | Keep all right records, add left info where available | 3 rows (Alice, Bob, Operations) |
| **Full Outer Join** | `how="full"`, `how="outer"`, or `how="full_outer"` | All rows from both DataFrames | Both left & right (NULL for non-matches) | Get complete picture with all matches and mismatches | 4 rows (Alice, Bob, Charlie, Operations) |
| **Left Semi Join** | `how="left_semi"` | Left rows that have a match in right | Left only | Filter left table by existence in right (like WHERE EXISTS) | 2 rows (Alice, Bob) - no dept_name column |
| **Left Anti Join** | `how="left_anti"` | Left rows that DON'T have a match in right | Left only | Find orphan records (like WHERE NOT EXISTS) | 1 row (Charlie) - no dept_name column |
| **Cross Join** | `crossJoin()` or `how="cross"` | Cartesian product (all combinations) | Both left & right | Generate all possible combinations (use with caution!) | 9 rows (3 × 3 combinations) |

---

## Key Takeaways

### When to Use Each Join:

1. **Inner Join**: "Give me only the records that match in both tables"
   * Example: Employees with valid departments

2. **Left Join**: "Give me all records from the left table, add right info if available"
   * Example: All employees (whether or not they have a department)

3. **Right Join**: "Give me all records from the right table, add left info if available"
   * Example: All departments (whether or not they have employees)

4. **Full Outer Join**: "Give me everything from both tables, show me all matches and mismatches"
   * Example: Complete employee-department view for data quality checks

5. **Left Semi Join**: "Filter left table to rows that exist in right table (don't include right columns)"
   * Example: Employees who have departments (but I don't need the department details)

6. **Left Anti Join**: "Give me left table rows that DON'T exist in right table"
   * Example: Employees without departments (orphan records)

7. **Cross Join**: "Give me every possible combination of left and right rows"
   * Example: All possible employee-department assignment scenarios
   * **Warning**: Can produce huge datasets! Use sparingly.

---

## Syntax Patterns

```python
# Standard join syntax
df1.join(df2, on="column_name", how="join_type")

# Multiple columns join
df1.join(df2, on=["col1", "col2"], how="join_type")

# Join with different column names
df1.join(df2, df1.emp_dept_id == df2.dept_id, how="join_type")

# Cross join (special syntax)
df1.crossJoin(df2)
```

---

## NULL Handling

* **Inner, Left Semi, Left Anti**: No NULLs (only matched or filtered records)
* **Left Join**: NULLs in **right** columns for non-matching left rows
* **Right Join**: NULLs in **left** columns for non-matching right rows
* **Full Outer Join**: NULLs in **both** left and right columns for non-matching rows